# Prétraitement - Telco Customer Churn

Objectif : nettoyer les données, traiter les valeurs manquantes/outliers, encoder les variables, puis préparer le split train/test et le pipeline de transformation.

## 1. Import des bibliothèques

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

import os
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

## 2. Chargement et nettoyage

In [2]:
df = pd.read_csv("../data/raw/Telco-Customer-Churn.csv")

# customerID n'apporte aucune information prédictive -> on le retire
df = df.drop(columns=['customerID'])

# TotalCharges est mal typé (object) à cause d'espaces vides pour les nouveaux clients
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Vérification : ces lignes correspondent-elles à des clients avec tenure = 0 ?
print(df.loc[df['TotalCharges'].isna(), 'tenure'].unique())

[0]


**Décision :** les 11 valeurs manquantes de `TotalCharges` correspondent toutes à des clients avec `tenure = 0` (nouveaux clients, pas encore facturés). On remplace donc logiquement par 0 plutôt que par la médiane, qui fausserait l'interprétation.

In [3]:
df['TotalCharges'] = df['TotalCharges'].fillna(0)
print("Valeurs manquantes restantes :", df.isna().sum().sum())

# Encodage de la cible en binaire (0 = reste, 1 = churn)
df['Churn'] = (df['Churn'] == 'Yes').astype(int)
df['Churn'].value_counts(normalize=True)

Valeurs manquantes restantes : 0


Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64

## 3. Traitement des outliers

In [4]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

def count_outliers_iqr(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((series < lower) | (series > upper)).sum()

for col in num_cols:
    print(f"{col}: {count_outliers_iqr(df[col])} outliers (méthode IQR)")

tenure: 0 outliers (méthode IQR)
MonthlyCharges: 0 outliers (méthode IQR)
TotalCharges: 0 outliers (méthode IQR)


**Décision :** aucune variable ne présente d'outliers selon la méthode IQR (valeurs cohérentes : facturation, ancienneté). Aucune suppression/capping nécessaire ici.

## 4. Split et Pipeline

In [5]:
X = df.drop(columns=['Churn'])
y = df['Churn']

cat_cols = X.select_dtypes(include='object').columns.tolist()
print("Variables catégorielles :", cat_cols)
print("Variables numériques :", num_cols)

Variables catégorielles : ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Variables numériques : ['tenure', 'MonthlyCharges', 'TotalCharges']


/tmp/ipykernel_524/587778648.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include='object').columns.tolist()


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train :", X_train.shape, " Test :", X_test.shape)

Train : (5634, 19)  Test : (1409, 19)


In [7]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_cols)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Shape après transformation :", X_train_processed.shape)

Shape après transformation : (5634, 29)


**Pourquoi ce choix ?**
- `StandardScaler` sur les variables numériques : nécessaire pour les modèles sensibles à l'échelle (Logistic Regression, SVM)
- `OneHotEncoder` sur les variables catégorielles : transforme chaque catégorie en colonne binaire, `drop='first'` évite la colinéarité parfaite
- Le préprocesseur est **fit uniquement sur le train** puis appliqué (`transform`) sur le test, pour éviter toute fuite de données (data leakage)

## 5. Sauvegarde

In [8]:
# Sauvegarde des données prétraitées (format dense pour simplicité)
pd.DataFrame(X_train_processed.toarray() if hasattr(X_train_processed, 'toarray') else X_train_processed)\
    .to_csv("../data/processed/X_train.csv", index=False)
pd.DataFrame(X_test_processed.toarray() if hasattr(X_test_processed, 'toarray') else X_test_processed)\
    .to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

# Sauvegarde du préprocesseur (pour pouvoir transformer de nouvelles données plus tard)
joblib.dump(preprocessor, "../models/preprocessor.pkl")

print("Fichiers sauvegardés dans data/processed/ et models/")

Fichiers sauvegardés dans data/processed/ et models/


## Conclusion du prétraitement
- Données nettoyées (0 valeur manquante restante), cible encodée en binaire
- Pas d'outliers significatifs à traiter
- Pipeline numérique (StandardScaler) + catégoriel (OneHotEncoder) construit et sauvegardé
- Jeux de données train/test prêts pour la modélisation

Prochaine étape : notebook `03_modeling_draft.ipynb` (entraînement et comparaison des modèles)